# FraudCast — Unified Fraud Detection Pipeline

Three detection stages combined into a single `fraud_score ∈ [0, 1]`:

| Stage | Method | Signal |
|---|---|---|
| 1 — Anomaly | Isolation Forest | Unusual amount / time-of-day |
| 2 — Location | Physics check (Haversine) | Impossible travel speed — no API needed |
| 3 — Classifier | Adaptive Weighted Fusion (GBM + NB + SVC) | 13 behavioural features |

Evaluation uses a **time-based train/test split** so the model is tested on data it never saw during training, mirroring real deployment.

In [ ]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt
from sklearn.ensemble import IsolationForest, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'potoro.csv'  # change to '/content/potoro.csv' in Colab

In [ ]:
# ── Load & Parse ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
# dayfirst=True handles the DD/MM/YY HH:MM format in potoro.csv
df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True)
df = df.sort_values('Timestamp').reset_index(drop=True)

ts_min  = df['Timestamp'].min().date()
ts_max  = df['Timestamp'].max().date()
years   = df['Timestamp'].dt.year.value_counts().sort_index().to_dict()

print(f'Loaded {len(df)} transactions  |  {ts_min} to {ts_max}')
print('Year distribution:', years)
print('Domains:', df['Domain'].value_counts().to_dict())
print('Labels:', df['Classification'].value_counts().to_dict())

## Fraud Synthesis

The real dataset has **zero fraud labels** — it's a personal transaction history. We inject 80 synthetic fraud transactions using four realistic attack patterns:

- **amount_spike** — transaction amount 6–15× above domain average  
- **location_jump** — transaction placed in Delhi or Bengaluru minutes after a Mumbai transaction  
- **off_hours** — otherwise-normal transaction moved to 1–5 am  
- **new_recipient** — large payment to an unknown vendor  

In [ ]:
def synthesize_fraud(df, n=80, seed=42):
    rng         = np.random.default_rng(seed)
    domain_mean = df.groupby('Domain')['Amount'].mean()
    patterns    = ['amount_spike', 'location_jump', 'off_hours', 'new_recipient']
    probs       = [0.30, 0.25, 0.25, 0.20]
    fraud_rows  = []

    city_coords = np.array([[28.6448, 77.2167],   # Delhi
                             [12.9716, 77.5946]])  # Bengaluru

    for _ in range(n):
        pat  = rng.choice(patterns, p=probs)
        base = df.sample(1, random_state=int(rng.integers(10000))).iloc[0].copy()

        if pat == 'amount_spike':
            m = float(domain_mean.get(base['Domain'], base['Amount']))
            base['Amount'] = m * rng.uniform(6, 15)

        elif pat == 'location_jump':
            # Teleport to Delhi or Bengaluru — physically impossible from Mumbai
            city = city_coords[rng.integers(2)]
            base['Latitude']  = float(city[0]) + rng.uniform(-0.3, 0.3)
            base['Longitude'] = float(city[1]) + rng.uniform(-0.3, 0.3)
            base['Location']  = 'Unknown'
            # Place 15–60 min after a random real transaction
            anchor = df.iloc[int(rng.integers(1, len(df)))]
            base['Timestamp'] = anchor['Timestamp'] + pd.Timedelta(minutes=int(rng.integers(15, 61)))

        elif pat == 'off_hours':
            ts = base['Timestamp']
            base['Timestamp'] = ts.replace(hour=int(rng.integers(1, 6)),
                                            minute=int(rng.integers(0, 60)))

        elif pat == 'new_recipient':
            base['Recepient'] = f'Unknown_Vendor_{rng.integers(9000, 9999)}'
            base['Amount']    = float(rng.uniform(800, 3500))

        base['Classification'] = 'Fraud'
        fraud_rows.append(base)

    return pd.DataFrame(fraud_rows)


fraud_df = synthesize_fraud(df, n=80)
full_df  = pd.concat([df, fraud_df], ignore_index=True)
full_df  = full_df.sort_values('Timestamp').reset_index(drop=True)

legit_n = int((full_df['Classification'] == 'Legit').sum())
fraud_n = int((full_df['Classification'] == 'Fraud').sum())
print(f'Full dataset: {len(full_df)} rows  |  Legit: {legit_n}  |  Fraud: {fraud_n} ({fraud_n/len(full_df)*100:.1f}%)')

## Feature Engineering

13 features built with an **expanding window** — each transaction only uses information available at the time it occurred (no look-ahead leakage).

| Feature | Description |
|---|---|
| `amount_zscore` | (amount − domain mean) / domain std |
| `amount_ratio` | amount / domain mean |
| `hour` | hour of day (0–23) |
| `is_night` | 1 if hour ∈ [22–5] |
| `is_weekend` | 1 if Saturday or Sunday |
| `recipient_is_new` | 1 if recipient never seen before |
| `recipient_freq` | count of prior payments to this recipient |
| `location_is_usual` | 1 if location in historical location set |
| `dist_from_last_km` | Haversine distance from previous transaction |
| `time_gap_min` | minutes since previous transaction |
| `implied_speed_kmh` | dist / time — flags location impossibility |
| `txn_count_1h` | number of transactions in previous hour |
| `txn_count_24h` | number of transactions in previous 24 h |

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2 - lat1) / 2)**2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2)**2
    return R * 2 * asin(sqrt(a))


def build_features(df):
    df   = df.sort_values('Timestamp').reset_index(drop=True)
    legit = df[df['Classification'] == 'Legit']

    # Domain baselines from legit transactions only
    domain_stats = legit.groupby('Domain')['Amount'].agg(['mean', 'std']).fillna(1)
    usual_locs   = set(legit['Location'].unique())
    rows         = []

    for i in range(len(df)):
        row  = df.iloc[i]
        prev = df.iloc[:i]
        ts   = row['Timestamp']

        # Amount features
        if row['Domain'] in domain_stats.index:
            d_mean = domain_stats.loc[row['Domain'], 'mean']
            d_std  = max(domain_stats.loc[row['Domain'], 'std'], 1.0)
        else:
            d_mean, d_std = row['Amount'], 1.0
        amt_z     = (row['Amount'] - d_mean) / d_std
        amt_ratio = row['Amount'] / d_mean if d_mean > 0 else 1.0

        # Time features
        hour       = ts.hour
        is_night   = int(hour >= 22 or hour <= 5)
        is_weekend = int(ts.dayofweek >= 5)

        # Recipient features (expanding window — only history up to this point)
        prev_rec     = prev['Recepient'].tolist() if len(prev) > 0 else []
        recip_is_new = int(row['Recepient'] not in prev_rec)
        recip_freq   = prev_rec.count(row['Recepient'])

        # Location features
        loc_is_usual = int(row['Location'] in usual_locs)
        if len(prev) > 0:
            last      = prev.iloc[-1]
            dist_km   = haversine(row['Latitude'],  row['Longitude'],
                                   last['Latitude'], last['Longitude'])
            gap_min   = (ts - last['Timestamp']).total_seconds() / 60.0
            speed_kmh = dist_km / (gap_min / 60.0) if gap_min > 0 else 0.0
        else:
            dist_km = gap_min = speed_kmh = 0.0

        # Velocity features (expanding window)
        if len(prev) > 0:
            txn_1h  = len(prev[prev['Timestamp'] >= ts - pd.Timedelta(hours=1)])
            txn_24h = len(prev[prev['Timestamp'] >= ts - pd.Timedelta(hours=24)])
        else:
            txn_1h = txn_24h = 0

        rows.append({
            'amount_zscore':     amt_z,
            'amount_ratio':      amt_ratio,
            'hour':              hour,
            'is_night':          is_night,
            'is_weekend':        is_weekend,
            'recipient_is_new':  recip_is_new,
            'recipient_freq':    recip_freq,
            'location_is_usual': loc_is_usual,
            'dist_from_last_km': dist_km,
            'time_gap_min':      gap_min,
            'implied_speed_kmh': min(speed_kmh, 500.0),
            'txn_count_1h':      txn_1h,
            'txn_count_24h':     txn_24h,
        })

    return pd.DataFrame(rows)


print('Computing features (expanding window, ~15 s)...')
feat_df = build_features(full_df)
labels  = (full_df['Classification'] == 'Fraud').astype(int).values
print(f'Done. Feature matrix: {feat_df.shape}')
print(feat_df.describe().round(2))

## Time-Based Train / Test Split

Training on transactions before July 2023, testing on July 2023 onward. This prevents any look-ahead leakage and tests whether the model generalises to new time periods — the same condition it faces in production.

In [ ]:
SPLIT_DATE = pd.Timestamp('2023-07-01')

train_mask = (full_df['Timestamp'] < SPLIT_DATE).values
test_mask  = ~train_mask

X_train, X_test = feat_df[train_mask], feat_df[test_mask]
y_train, y_test = labels[train_mask],  labels[test_mask]

assert y_train.sum() > 0, 'No fraud in train set — adjust SPLIT_DATE or increase n_fraud'
assert y_test.sum()  > 0, 'No fraud in test set  — adjust SPLIT_DATE or increase n_fraud'

test_start = full_df.loc[test_mask, 'Timestamp'].min().date()
print(f'Train: {train_mask.sum():>4} rows  |  fraud: {y_train.sum()}')
print(f'Test:  {test_mask.sum():>4} rows  |  fraud: {y_test.sum()}  (starting {test_start})')

## Stage 1 — Isolation Forest (Anomaly Score)

Trained on **legit-only** training data. Outputs a score in [0, 1] where 1 = most anomalous. Uses 5 statistical features: amount z-score, amount ratio, hour, is_night, txn_count_1h.

In [ ]:
ANOMALY_FEATURES = ['amount_zscore', 'amount_ratio', 'hour', 'is_night', 'txn_count_1h']


class AnomalyStage:
    def __init__(self, contamination=0.1):
        self.model  = IsolationForest(contamination=contamination, random_state=42)
        self.scaler = MinMaxScaler()
        self._s_min = self._s_range = None

    def fit(self, X_legit):
        X   = self.scaler.fit_transform(X_legit[ANOMALY_FEATURES].values)
        self.model.fit(X)
        raw = self.model.decision_function(X)
        self._s_min   = raw.min()
        self._s_range = raw.max() - raw.min() + 1e-9

    def score(self, X):
        Xs  = self.scaler.transform(X[ANOMALY_FEATURES].values)
        raw = self.model.decision_function(Xs)
        # More negative decision_function = more anomalous; flip and normalise to [0, 1]
        return np.clip(1 - (raw - self._s_min) / self._s_range, 0, 1)


anomaly = AnomalyStage(contamination=0.1)
anomaly.fit(X_train[y_train == 0])   # legit-only training

train_if = anomaly.score(X_train)
test_if  = anomaly.score(X_test)
print(f'IF scores on test  — fraud: {test_if[y_test==1].mean():.3f}  legit: {test_if[y_test==0].mean():.3f}')

## Stage 2 — Location Feasibility (Physics Check)

If the implied travel speed between consecutive transactions exceeds 150 km/h, the location jump is physically suspicious. No external API needed — just Haversine distance and timestamp difference.

The `implied_speed_kmh` feature was already computed in `build_features`.

In [ ]:
SPEED_CEILING = 150  # km/h — above this, score rises toward 1.0


def location_score(X):
    return np.clip(X['implied_speed_kmh'].values / SPEED_CEILING, 0, 1)


train_loc = location_score(X_train)
test_loc  = location_score(X_test)
print(f'Loc scores on test — fraud: {test_loc[y_test==1].mean():.3f}  legit: {test_loc[y_test==0].mean():.3f}')

## Stage 3 — Adaptive Weighted Fusion Classifier

Ensemble of three classifiers whose prediction weights are derived from their validation-set accuracy. SMOTE is applied to the training set to counteract class imbalance before fitting.

In [ ]:
class AdaptiveWeightedFusion:
    def __init__(self):
        self.clfs = [
            GradientBoostingClassifier(n_estimators=100, random_state=42),
            GaussianNB(),
            SVC(probability=True, random_state=42),
        ]
        self.scaler  = MinMaxScaler()
        self.weights = None

    def fit(self, X_tr, y_tr, X_val, y_val):
        from sklearn.metrics import accuracy_score
        Xs   = self.scaler.fit_transform(X_tr)
        Xvs  = self.scaler.transform(X_val)
        accs = []
        for clf in self.clfs:
            clf.fit(Xs, y_tr)
            accs.append(accuracy_score(y_val, clf.predict(Xvs)))
        total        = sum(accs) or 1.0
        self.weights = [a / total for a in accs]
        for name, w in zip(['GBC', 'NB', 'SVC'], self.weights):
            print(f'  {name}: {w:.3f}', end='  ')
        print()

    def predict_proba(self, X):
        Xs    = self.scaler.transform(X)
        probs = [clf.predict_proba(Xs) for clf in self.clfs]
        return sum(w * p for w, p in zip(self.weights, probs))


# SMOTE on training data to balance Fraud / Legit
smote       = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_train.values, y_train)

X_sm_tr, X_sm_val, y_sm_tr, y_sm_val = train_test_split(
    X_sm, y_sm, test_size=0.2, stratify=y_sm, random_state=42)

print('AWF weights:')
awf = AdaptiveWeightedFusion()
awf.fit(X_sm_tr, y_sm_tr, X_sm_val, y_sm_val)

train_clf = awf.predict_proba(X_train.values)[:, 1]
test_clf  = awf.predict_proba(X_test.values)[:, 1]
print(f'CLF scores on test — fraud: {test_clf[y_test==1].mean():.3f}  legit: {test_clf[y_test==0].mean():.3f}')

## Unified FraudScore Pipeline

All three stage outputs are combined with fixed weights into a single score. The weights reflect how reliable each signal is:

```
fraud_score = 0.25 * anomaly_score + 0.20 * location_score + 0.55 * classifier_score
```

A transaction is flagged as fraud if `fraud_score >= THRESHOLD` (default 0.35).

In [ ]:
W_IF  = 0.25
W_LOC = 0.20
W_CLF = 0.55
THRESHOLD = 0.30


def fraud_score(if_s, loc_s, clf_s):
    return W_IF * if_s + W_LOC * loc_s + W_CLF * clf_s


train_scores = fraud_score(train_if, train_loc, train_clf)
test_scores  = fraud_score(test_if,  test_loc,  test_clf)
test_preds   = (test_scores >= THRESHOLD).astype(int)


# ── Per-Transaction Inference Demo ────────────────────────────────────────────

def predict_transaction(amount, timestamp_str, domain, recipient, lat, lon,
                         prev_lat=None, prev_lon=None, prev_ts_str=None):
    ts     = pd.to_datetime(timestamp_str, dayfirst=True)
    d_data = df[df['Domain'] == domain]['Amount']
    d_mean = d_data.mean() if len(d_data) > 0 else amount
    d_std  = max(d_data.std(), 1.0) if len(d_data) > 1 else 1.0

    if prev_lat is not None and prev_lon is not None and prev_ts_str is not None:
        prev_ts  = pd.to_datetime(prev_ts_str, dayfirst=True)
        dist_km  = haversine(lat, lon, prev_lat, prev_lon)
        gap_min  = (ts - prev_ts).total_seconds() / 60.0
        speed    = dist_km / (gap_min / 60.0) if gap_min > 0 else 0.0
    else:
        dist_km = gap_min = speed = 0.0

    txn = pd.DataFrame([{
        'amount_zscore':     (amount - d_mean) / d_std,
        'amount_ratio':      amount / d_mean if d_mean > 0 else 1.0,
        'hour':              ts.hour,
        'is_night':          int(ts.hour >= 22 or ts.hour <= 5),
        'is_weekend':        int(ts.dayofweek >= 5),
        'recipient_is_new':  int(recipient not in df['Recepient'].values),
        'recipient_freq':    int((df['Recepient'] == recipient).sum()),
        'location_is_usual': int(dist_km < 30),
        'dist_from_last_km': dist_km,
        'time_gap_min':      gap_min,
        'implied_speed_kmh': min(speed, 500.0),
        'txn_count_1h':      0,
        'txn_count_24h':     0,
    }])

    if_s  = anomaly.score(txn)[0]
    loc_s = location_score(txn)[0]
    clf_s = awf.predict_proba(txn.values)[0, 1]
    f_s   = fraud_score(if_s, loc_s, clf_s)

    verdict = 'FRAUD' if f_s >= THRESHOLD else 'LEGIT'
    print(f'  Anomaly score:    {if_s:.3f}')
    print(f'  Location score:   {loc_s:.3f}')
    print(f'  Classifier score: {clf_s:.3f}')
    print(f'  Fraud score:      {f_s:.3f}  ->  {verdict}')
    return f_s


print('--- Normal (McDonalds, Rs 120, same area) ---')
predict_transaction(120, '2024-01-15 14:30:00', 'Food', 'McDonalds',
                    19.0760, 72.8777, 19.0760, 72.8777, '2024-01-15 12:00:00')

print('\n--- Location jump (Delhi, 20 min after Mumbai txn) ---')
predict_transaction(500, '2024-01-15 14:30:00', 'Food', 'McDonalds',
                    28.6448, 77.2167, 19.0760, 72.8777, '2024-01-15 14:10:00')

print('\n--- Amount spike (Rs 25000 Retail) ---')
predict_transaction(25000, '2024-01-15 14:30:00', 'Retail', 'Unknown Store',
                    19.0760, 72.8777, 19.0760, 72.8777, '2024-01-15 14:25:00')

print('\n--- Off-hours + new recipient (3am) ---')
predict_transaction(2500, '2024-01-15 03:15:00', 'Retail', 'Unknown_Vendor_9999',
                    19.0760, 72.8777, 19.0760, 72.8777, '2024-01-15 03:10:00')

## Evaluation

Time-based test set — model sees only post-split transactions.

In [ ]:
print('=== Classification Report (test set) ===')
print(classification_report(y_test, test_preds, target_names=['Legit', 'Fraud'], digits=3))

roc_auc = roc_auc_score(y_test, test_scores)
print(f'ROC-AUC: {roc_auc:.3f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Confusion matrix
cm = confusion_matrix(y_test, test_preds)
ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, test_scores)
axes[1].plot(fpr, tpr, color='steelblue', label=f'AUC = {roc_auc:.2f}')
axes[1].plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Score distribution
axes[2].hist(test_scores[y_test == 0], bins=30, alpha=0.6, label='Legit', color='steelblue')
axes[2].hist(test_scores[y_test == 1], bins=20, alpha=0.7, label='Fraud', color='tomato')
axes[2].axvline(THRESHOLD, color='black', linestyle='--', label=f'Threshold = {THRESHOLD}')
axes[2].set_xlabel('Fraud Score')
axes[2].set_ylabel('Count')
axes[2].set_title('Score Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('fraudcast_eval.png', dpi=150, bbox_inches='tight')
plt.show()